### Code 124M GPT-2 From scratch

In [346]:
import torch
import torch.nn as nn

In [347]:
cfg={
    'emb_dim':768,
    'num_heads':12,
    'context_length':768,
    'vocab_size':50257,
    'drop_rate':0.1,
    'n_layers':12

}

In [348]:
nn.Parameter(torch.ones(cfg['emb_dim']))
        

Parameter containing:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1.

In [349]:
x=10e-5
x

0.0001

In [350]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.epochs=10e-5
        self.shift=nn.Parameter(torch.zeros(emb_dim))
        self.scale=nn.Parameter(torch.ones(emb_dim))
        

    def forward(self,x):
        mean=x.mean(keepdim=True,dim=-1)
        var=x.var(keepdim=True,dim=-1)
        new_x=(x-mean)/(var+self.epochs)
        return new_x*self.scale + self.shift

In [351]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads ==0 ),\
            "d_out must be divisible by num_heads"
        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads

        self.w_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_value=nn.Linear(d_in,d_out,bias=qkv_bias)

        self.out_proj=nn.Linear(d_out,d_out)  ## linear layer of combine head output
        self.dropout=nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length),
                       diagonal=1)

        )

    def forward(self,x):
        b,num_tokens,d_in=x.shape 

        keys=self.w_key(x) # shape:b,num_token,d_in
        values=self.w_value(x)
        query=self.w_query(x)

        ## we split matrix 
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b,num_tokens,self.num_heads,self.head_dim)
        query=query.view(b,num_tokens,self.num_heads,self.head_dim)

        ## transpose the matrix 

        keys=keys.transpose(1,2)
        values=values.transpose(1,2)
        query=query.transpose(1,2)

        ## dot product of attention with casual mask 
        att_scores=query@keys.transpose(2,3) # each head dot product
        
        ## mask the numbers of tokens and converted into bool
        mask_bool=self.mask.bool()[:num_tokens, :num_tokens]

        ## use mask to fill attention score 
        att_scores.masked_fill(mask_bool,-torch.inf)

        att_weights=torch.softmax(att_scores/keys.shape[-1]**0.5,dim=-1)
        att_weights=self.dropout(att_weights)

        # shape (b,num_tokens,num_heads,head_dim)
        context_vec=(att_weights@values).transpose(1,2)

        # combine heads , d_out=num_head*head_dim
        context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
        context_vec=self.out_proj(context_vec)

        return context_vec


In [352]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,x):
        return x*0.5*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*
                      (x+0.4475*torch.pow(x,3))))

In [353]:
class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg['emb_dim'],4*cfg['emb_dim']),
            GELU(),
            nn.Linear(4*cfg['emb_dim'],cfg['emb_dim'])
        )

    def forward(self,x):
        return self.layers(x)


In [354]:
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.att=MultiHeadAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_length=cfg['context_length'],
            dropout=cfg['drop_rate'],
            num_heads=cfg['num_heads'],
        )
        self.ff=FeedForward(cfg)
        self.norm1=LayerNorm(cfg['emb_dim'])
        self.norm2=LayerNorm(cfg['emb_dim'])
        self.drop_shortcut=nn.Dropout(cfg['drop_rate'])

    def forward(self,x):
        x_shortcut=x
        x=self.norm1(x)
        x=self.att(x)
        x=self.drop_shortcut(x)
        x=x+x_shortcut ## shortcut connection

        x_shortcut=x

        x=self.norm2(x)
        x=self.ff(x)
        x=self.drop_shortcut(x)

        x=x+x_shortcut ## shortcut connection

        return x

In [355]:
class GPTModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        self.pos_emb=nn.Embedding(cfg['context_length'],cfg['emb_dim'])
        self.drop_rate=nn.Dropout(cfg["drop_rate"])

        self.trf_block=nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])] ## 12 transformer block sequential stack
        )
        self.final_norm=LayerNorm(cfg['emb_dim'])
        self.out_head=nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)


    def forward(self,in_idx):
            batch_size,seq_len = in_idx.shape
            tok_embeds=self.tok_emb(in_idx) # tok_id -> tok_emb
            pos_embeds=self.pos_emb(torch.arange(seq_len,device=in_idx.device))
            x=tok_embeds+pos_embeds # batch,num_tok , emb_dim
            x=self.drop_rate(x)
            x=self.trf_block(x)
            x=self.final_norm(x)
            logits=self.out_head(x)
            return logits


In [356]:
in_idx=torch.tensor([[12,42,13], ## token ids not converted into the emb
                    [53,25,14]]) # 3=seq len ,2=batch

In [357]:
in_idx.shape

torch.Size([2, 3])

In [358]:
x,y=in_idx.shape

In [359]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [360]:
torch.manual_seed(123)
model = GPTModel(cfg)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.0365, -0.1077, -0.2191,  ..., -0.1109,  0.3669,  0.6952],
         [-0.2112,  0.6087, -0.1273,  ...,  0.1099,  0.3269,  0.1551],
         [ 0.5831, -0.1987,  0.1683,  ..., -0.0439, -0.5337,  0.3596],
         [-0.0892, -0.0475,  0.0130,  ...,  0.0913,  0.2850,  0.6844]],

        [[ 0.2589, -0.1236, -0.2747,  ..., -0.0644,  0.0350,  0.7789],
         [-0.6447,  0.6781,  0.1367,  ...,  0.3473,  0.3117, -0.2458],
         [ 0.3103,  0.5187,  0.6748,  ...,  0.1712, -0.0690,  0.2112],
         [-0.1391,  0.1216,  0.7947,  ...,  0.4323, -0.2433,  0.0483]]],
       grad_fn=<UnsafeViewBackward0>)


In [364]:
out.max(dim=-1)

torch.return_types.max(
values=tensor([[1.6290, 1.5427, 1.5790, 1.3808],
        [1.4397, 1.5542, 1.5365, 1.4932]], grad_fn=<MaxBackward0>),
indices=tensor([[48571, 15519, 49385, 48599],
        [49845, 31138,   371, 20199]]))

In [365]:
out.min(dim=-1)

torch.return_types.min(
values=tensor([[-1.5578, -1.3804, -1.4517, -1.5729],
        [-1.3507, -1.5163, -1.4487, -1.7324]], grad_fn=<MinBackward0>),
indices=tensor([[10300, 29136, 26056,  7537],
        [30167, 48253,  9746, 37629]]))

In [367]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Total number of parameters: 162,812,928
Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


In [368]:
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Number of trainable parameters considering weight tying: 124,215,552


In [371]:
total_size_bytes = total_params * 4 #A
total_size_bytes_gpt2 = total_params_gpt2 * 4 #A
total_size_mb = total_size_bytes / (1024 * 1024) #B
total_size_mb_gpt2 = total_size_bytes_gpt2 / (1024 * 1024) #B
print(f"Total size of the model: {total_size_mb:.2f} MB")
print(f"Total size of the model gpt2: {total_size_mb_gpt2:.2f} MB")

Total size of the model: 621.08 MB
Total size of the model gpt2: 473.84 MB
